# Assignment 4 — Hospital Privacy Dashboard

Information Security (CS-3002), Muddassir Rizwan (23K-0784)

This notebook walks through the same code the dashboard uses, step by step. It checks the three parts of the CIA triad — keeping data private, keeping it correct, and keeping the system working — plus the GDPR bits: consent, anonymize and decrypt, old records, and erasing a patient.


## Step 1: Make the database

`init_db()` makes `hospital.db` with the users, patients, logs and settings tables, and adds two triggers so log entries cannot be changed or deleted once written.


In [1]:
from database import get_all_patients, get_connection, init_db

init_db()

patients = get_all_patients()
print(f"Total sample patients: {len(patients)}")
for p in patients:
    print(f"  #{p[0]} | {p[1]} | {p[2]} | {p[3]} | {p[4]} | {p[5]} | {p[6]}")

conn = get_connection()
triggers = [r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='trigger'").fetchall()]
conn.close()
print("Active SQLite integrity triggers:", triggers)

Total sample patients: 8
  #1 | Ahmed Khan | 0300-1234567 | Flu | ANON_1796 | XXX-XXX-4567 | 2026-11-20
  #2 | Sara Ahmed | 0321-7654321 | Diabetes | ANON_5858 | XXX-XXX-4321 | 2026-11-21
  #3 | Bilal Hussain | 0333-4567890 | Fracture | ANON_9774 | XXX-XXX-7890 | 2026-11-22
  #4 | Ayesha Malik | 0345-9876543 | Asthma | ANON_3394 | XXX-XXX-6543 | 2026-11-23
  #5 | Usman Ali | 0301-2345678 | Fever | ANON_7318 | XXX-XXX-5678 | 2026-11-24
  #6 | Fatima Noor | 0322-3456789 | Migraine | ANON_1617 | XXX-XXX-6789 | 2026-11-25
  #7 | Hassan Raza | 0312-8877665 | Hypertension | ANON_8960 | XXX-XXX-7665 | 2025-12-15
  #8 | Zainab Tariq | 0331-5544332 | Seasonal Allergy | ANON_9749 | XXX-XXX-4332 | 2025-06-10
Active SQLite integrity triggers: ['prevent_log_update', 'prevent_log_delete']


## Step 2: Login and roles

The demo accounts from the assignment are `admin`, `Dr. Bob` and `Alice_recep`. Passwords are stored as salted hashes, not plain text. The database checks the role itself, so it is not just the buttons that hide things.


In [2]:
from auth import can_do, check_login
from database import anonymize_all, get_all_logs

for uname, pw in [("admin", "admin123"), ("Dr. Bob", "doc123"), ("Alice_recep", "rec123"), ("admin", "wrongpw")]:
    res = check_login(uname, pw)
    print(f"{uname:12s} -> {res['role'] if res else 'LOGIN FAILED'}")

print("\nRBAC summary:")
for r in ["admin", "doctor", "receptionist"]:
    print(f"  {r:12s} | raw={can_do(r, 'view_raw')} | diag={can_do(r, 'view_diagnosis')} | add={can_do(r, 'add')} | anonymize={can_do(r, 'anonymize')} | logs={can_do(r, 'view_logs')}")

# Backend RBAC check: Doctor attempting to open audit logs or anonymize must raise PermissionError
for fn_name, fn in [("get_all_logs", lambda: get_all_logs(role="doctor")), ("anonymize_all", lambda: anonymize_all(role="receptionist"))]:
    try:
        fn()
    except PermissionError as err:
        print(f"Blocked unauthorized {fn_name}: {err}")

admin        -> admin


Dr. Bob      -> doctor


Alice_recep  -> receptionist


admin        -> LOGIN FAILED

RBAC summary:
  admin        | raw=True | diag=True | add=True | anonymize=True | logs=True
  doctor       | raw=False | diag=True | add=False | anonymize=False | logs=False
  receptionist | raw=False | diag=False | add=True | anonymize=False | logs=False
Blocked unauthorized get_all_logs: Role 'doctor' is not authorized for 'view_logs'.
Blocked unauthorized anonymize_all: Role 'receptionist' is not authorized for 'anonymize'.


## Step 3: Masking, and what each role sees

Names become ANON ids, contacts show only the last four digits, and the receptionist sees the diagnosis as `***`. The doctor sees the diagnosis in full, because that is the medical record they need.


In [3]:
from crypto_utils import make_anon_name, mask_contact, mask_diagnosis, mask_name
from database import get_patients_for_role

print("Masking helpers:")
print("  mask_name('Ahmed Khan')        ->", mask_name("Ahmed Khan"))
print("  make_anon_name('Ahmed Khan',1) ->", make_anon_name("Ahmed Khan", 1))
print("  mask_contact('0300-1234567')   ->", mask_contact("0300-1234567"))
print("  mask_diagnosis('Flu')          ->", mask_diagnosis("Flu"))

print("\nWhat each role receives for Patient #1:")
for r in ["admin", "doctor", "receptionist"]:
    row = get_patients_for_role(r)[0]
    print(f"  {r:12s} -> {row}")

Masking helpers:
  mask_name('Ahmed Khan')        -> ***
  make_anon_name('Ahmed Khan',1) -> ANON_1796
  mask_contact('0300-1234567')   -> XXX-XXX-4567
  mask_diagnosis('Flu')          -> ***

What each role receives for Patient #1:
  admin        -> {'patient_id': 1, 'name': 'Ahmed Khan', 'contact': '0300-1234567', 'diagnosis': 'Flu', 'anonymized_name': 'ANON_1796', 'anonymized_contact': 'XXX-XXX-4567', 'encrypted_preview': 'gAAAAABrCr1_zw8lY0At7z...', 'date_added': '2026-11-20', 'days_old': 8, 'days_left': 357, 'retention_status': 'Active (357d left)', 'consent_given': True, 'is_erased': False}
  doctor       -> {'patient_id': 1, 'anonymized_name': 'ANON_1796', 'anonymized_contact': 'XXX-XXX-4567', 'diagnosis': 'Flu', 'date_added': '2026-11-20', 'days_old': 8, 'retention_status': 'Active (357d left)', 'is_erased': False}
  receptionist -> {'patient_id': 1, 'anonymized_name': 'ANON_1796', 'anonymized_contact': 'XXX-XXX-4567', 'diagnosis': '***', 'date_added': '2026-11-20', 'days_old':

## Step 4: Anonymize and decrypt with Fernet

Anonymize replaces the real name and contact in the database with masked values. The real values only live in the encrypted columns, and decrypt brings them back for the admin. The key is in `secret.key`, which is never pushed to the repo.


In [4]:
from crypto_utils import decrypt_text
from database import anonymize_all, decrypt_all, get_encrypted_sample

n_anon = anonymize_all(role="admin")
print(f"Anonymized & encrypted {n_anon} records in SQLite.")
sample = get_encrypted_sample()
print(f"DB row #1 while anonymized -> name='{sample[1]}', anon='{sample[2]}', contact='{sample[3]}'")
print(f"Fernet ciphertext          -> {sample[4][:60]}...")
print(f"Direct Fernet decrypt test -> {decrypt_text(sample[4])}")

n_dec = decrypt_all(role="admin")
restored = get_all_patients()[0]
print(f"Decrypted {n_dec} records -> DB row #1 restored: name='{restored[1]}', contact='{restored[2]}'")

Anonymized & encrypted 8 records in SQLite.
DB row #1 while anonymized -> name='ANON_1796', anon='ANON_1796', contact='XXX-XXX-4567'
Fernet ciphertext          -> gAAAAABrCr1_L1JpxeSJ7XKB28sRuBRur0pfbSHRKeDgWr7FRy02EuNcGgIP...
Direct Fernet decrypt test -> Ahmed Khan


Decrypted 8 records -> DB row #1 restored: name='Ahmed Khan', contact='0300-1234567'


## Step 5: Consent, blind edit, and the audit log

Adding a patient without consent is blocked. When the receptionist edits a contact, the masked and encrypted columns update too, and the diagnosis stays as it is. Trying to delete a log entry fails, because the trigger blocks it.


In [5]:
import sqlite3
from database import add_patient, get_all_logs, log_action, update_patient

# 1. Consent enforcement check
try:
    add_patient("No Consent", "0300-1112222", "Cold", "2026-11-28", consent_given=False, role="receptionist")
except ValueError as err:
    print("Consent block works:", err)

# 2. Add patient with consent + Receptionist blind edit (name blank, new contact, diagnosis ignored)
pid = add_patient("Test Patient", "0300-9991111", "Cold", "2026-11-28", consent_given=True, role="receptionist")
log_action(3, "receptionist", "add", f"Added patient #{pid} (consent verified)")
changed = update_patient(pid, name="", contact="0300-9992222", diagnosis="ShouldBeIgnored", role="receptionist")
log_action(3, "receptionist", "update", f"Updated patient #{pid} (fields: {', '.join(changed)})")

updated_row = [p for p in get_all_patients() if p[0] == pid][0]
print(f"After blind edit -> changed={changed}, contact='{updated_row[2]}', masked='{updated_row[5]}', diagnosis='{updated_row[3]}'")

# Clean up temporary test patient
conn = get_connection()
conn.execute("DELETE FROM patients WHERE patient_id = ?", (pid,))
conn.commit()

# 3. Verify SQLite trigger blocks audit log deletion
try:
    conn.execute("DELETE FROM logs")
except sqlite3.IntegrityError as err:
    print("Immutable log trigger blocked DELETE:", err)
finally:
    conn.close()

print("Latest audit log entries:")
for entry in get_all_logs(role="admin")[:3]:
    print(" ", entry)

Consent block works: Patient consent is required under GDPR.
After blind edit -> changed=['contact'], contact='0300-9992222', masked='XXX-XXX-2222', diagnosis='Cold'
Immutable log trigger blocked DELETE: Audit logs are immutable and cannot be deleted
Latest audit log entries:
  (2, 3, 'receptionist', 'update', '2026-11-28 16:39:44', 'Updated patient #9 (fields: contact)')
  (1, 3, 'receptionist', 'add', '2026-11-28 16:39:44', 'Added patient #9 (consent verified)')


## Step 6: Backups and old records

The admin can download the patients and the logs as CSV files. Each record shows how old it is, and records older than the retention limit (365 days) are flagged, so they can be erased.


In [6]:
from database import export_logs_csv, export_patients_csv, get_actions_per_day, get_retention_days

print("Patients CSV (anonymized) preview:")
for line in export_patients_csv(anonymized=True, role="admin").splitlines()[:3]:
    print(" ", line)

print("\nAudit Logs CSV header:", export_logs_csv(role="admin").splitlines()[0])

print(f"\nGDPR Retention Status (policy = {get_retention_days()} days):")
for p in get_patients_for_role("admin"):
    print(f"  #{p['patient_id']} {p['anonymized_name']} | added {p['date_added']} | {p['days_old']:3d}d old | {p['retention_status']}")

print("\nActions per day:", get_actions_per_day(role="admin"))

Patients CSV (anonymized) preview:
  patient_id,name,contact,diagnosis,anonymized_name,anonymized_contact,date_added
  1,ANON_1796,XXX-XXX-4567,Flu,ANON_1796,XXX-XXX-4567,2026-11-20
  2,ANON_5858,XXX-XXX-4321,Diabetes,ANON_5858,XXX-XXX-4321,2026-11-21

Audit Logs CSV header: log_id,user_id,role,action,timestamp,details

GDPR Retention Status (policy = 365 days):
  #1 ANON_1796 | added 2026-11-20 |   8d old | Active (357d left)
  #2 ANON_5858 | added 2026-11-21 |   7d old | Active (358d left)
  #3 ANON_9774 | added 2026-11-22 |   6d old | Active (359d left)
  #4 ANON_3394 | added 2026-11-23 |   5d old | Active (360d left)
  #5 ANON_7318 | added 2026-11-24 |   4d old | Active (361d left)
  #6 ANON_1617 | added 2026-11-25 |   3d old | Active (362d left)
  #7 ANON_8960 | added 2025-12-15 | 348d old | Expiring Soon (17d left)
  #8 ANON_9749 | added 2025-06-10 | 536d old | Expired (171d over)

Actions per day: [('2026-11-28', 'add', 1), ('2026-11-28', 'update', 1)]


## How to run the app

```
pip install -r requirements.txt
streamlit run app.py
```

The app opens in the browser. The database file is made on the first run.

One thing to know if you run this notebook again: Steps 5 and 6 write to the log, and log entries cannot be deleted, so their outputs grow a little on every run. The patient data stays the same.
